## First look

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv('../data/train-test.csv')
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df.head()

Looks like a problem of predicting freight load prices from market history.
- load_id seems to be the main id
- posted_rate seems to be the target.
- pickup, delivery, pickup and delivery coordinates, equipment, weight, date, seem like feature candidates.
- distance may be related to the coordinates, although I am not sure if it is a routing-based calculation or a straight-line distance between the points; if it is the first case it can contain useful information that does not come from the coordinates.
- date can have useful information (prices change depending on specific dates or seasons),
- quote_signal I am not sure what it is
- hypothesis: market_index may not contain really useful information

## Initial exploration

In [ ]:
df.describe()   

In [ ]:
df.describe(include="object") 

In [ ]:
print(f'load_id. distinct: {df["load_id"].nunique()} duplicated: {df["load_id"].duplicated().sum()}')
print(f'Nulls: \n{df.isna().sum()}')
print(f'interval: {df["date"].min(), df["date"].max()}, {df["date"].max() - df["date"].min()}')

- load_id has only unique values, it is the main id.
- pickup, delivery and equipment are categorical, with 64, 64 and 3 classes respectively
- weight has 300 missing values, market_index 374
The period goes from January 1 2025 to October 31 2025, 303 days



## Numeric distributions and correlations

In [ ]:
df["posted_rate"].hist(bins=50)          

In [ ]:
df['posted_rate_log'] = np.log1p(df['posted_rate'])
df["posted_rate_log"].hist(bins=50)    

In [ ]:
df.plot.scatter(x="distance", y="posted_rate", alpha=0.1)
df.plot.scatter(x="weight", y="posted_rate", alpha=0.1)

In [ ]:
df.corr(numeric_only=True)["posted_rate"]  

In [ ]:
# great-circle distance in miles; compare with given distance column
def haversine_miles(lat1, lon1, lat2, lon2):
    r = 3959.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * r * np.arcsin(np.sqrt(a))

df["dist_hav"] = haversine_miles(
    df["pickup_lat"], df["pickup_lon"],
    df["delivery_lat"], df["delivery_lon"],
)

df.plot.scatter(x="dist_hav", y="distance", alpha=0.15)
plt.show()

df[["distance", "dist_hav"]].corr()

- the target shows a distribution with large outliers, log1p makes the distribution a bit more normal or uniform
- the coordinates explain almost all the information contained in distance according to the calculation
- distance is strongly correlated with the target

Correlations with the target
weight          0.034840
market_index    0.034165
quote_signal   -0.039858

- weight is very weakly correlated with the target, also there appear two big groups of weight scales, the second more concentrated
- market_index and quote_signal correlate with very small magnitudes, positive and negative respectively

Probably distance is the best predictor

## Distribution by categories

In [ ]:
df.boxplot(column="posted_rate", by="equipment")

In [ ]:
g = df.groupby("pickup")["posted_rate"].median().sort_values()
g.plot(kind="barh", figsize=(8, 10))

In [ ]:
g = df.groupby("delivery")["posted_rate"].median().sort_values()
g.plot(kind="barh", figsize=(8, 10))

In [ ]:
# city-level median rates: how similar are origin vs destination rankings?
med_pickup = df.groupby("pickup")["posted_rate"].median()
med_delivery = df.groupby("delivery")["posted_rate"].median()

cmp = pd.concat(
    {"pickup": med_pickup, "delivery": med_delivery},
    axis=1,
)

cmp["pickup"].corr(cmp["delivery"])

- the target with respect to equipment classes seems to be distributed uniformly, but dry van and reefer seem to have more large outliers than flatbed, although it probably does not affect too much
- the rankings of median rate by city are very similar in origin and destination role → there is a signal of "price level by city". Still it is reasonable to treat pickup and delivery as different features, because they describe different ends of the same trip.

## Temporal exploration

In [ ]:
# by day
daily = df.groupby("date")["posted_rate"].median()
daily.plot(figsize=(10, 4), title="median posted_rate by day")
plt.ylabel("posted_rate")
plt.show()

# by month
df.groupby(df["date"].dt.to_period("M"))["posted_rate"].median().plot(kind="bar", figsize=(8, 4))
plt.show()

- there are important variations of the average by day
- the averages do not vary much by month

## Validation check

In [ ]:
val = pd.read_csv("../data/validation.csv")
val["date"] = pd.to_datetime(val["date"])

val.describe()

In [ ]:
# count labels in validation that never appear in train-test
cat_cols = ["pickup", "delivery", "equipment"]

for col in cat_cols:
    train_set = set(df[col].dropna().unique())
    val_set = set(val[col].dropna().unique())
    print(col, "unseen in train:", len(val_set - train_set),
          "only in train:", len(train_set - val_set))


Validation
- has the same columns as train-test, weight and market_index also have missing values
- validation contains classes for pickup and delivery that train-test does not contain


## Updated hypotheses
hypothesis: the coordinates could explain pickup and delivery and generalize better than the labels themselves

In [ ]:
# max distinct lat/lon per city name; 1 means city and coords are 1:1
print("pickup", df.groupby("pickup")[["pickup_lat", "pickup_lon"]].nunique().max())
print("delivery", df.groupby("delivery")[["delivery_lat", "delivery_lon"]].nunique().max())

Each city has unique coordinates → pickup/delivery are redundant with their lat/lon. coordinates could be better to avoid redundancy and improve generalization.

## December input check

In [ ]:
dec = pd.read_csv("../data/december-chart-inputs.csv")
dec["date"] = pd.to_datetime(dec["date"])
dec.head()

In [ ]:
# time ranges: train-test, validation, december
print("train-test:", df["date"].min(), "→", df["date"].max())
print("validation:", val["date"].min(), "→", val["date"].max())
print("december:", dec["date"].min(), "→", dec["date"].max())


In [ ]:
# count labels in december that never appear in train-test

cat_cols = ["pickup", "delivery", "equipment"]

for col in cat_cols:
    train_set = set(df[col].dropna().unique())
    dec_set = set(dec[col].dropna().unique())
    print(col, "unseen in train:", len(dec_set - train_set),
          "only in train:", len(train_set - dec_set))

- december input does not include columns like the coordinates, so the model should take into account pickup and delivery cities (validation also has cities; december just has no lat/lon)
- december input only contains cities that are already in train
- train-test, validation and december are separated in time in sequence, so this suggests a temporal split is useful

## Takeaways

- Target: `posted_rate` (skewed; `log1p` helps the shape). Apparent dominant predictor: `distance`.
- Geo: city ↔ lat/lon is 1:1 (redundant); `distance` ≈ haversine. Preferring only lat/lon columns is not enough for december unless we derive coords from the city names.
- Pickup/delivery: still different ends of the trip; there is a city price-level effect, origin/destination profiles look similar.
- `equipment`: 3 classes, modest differences in the target.
- `weight`, `market_index`, `quote_signal`: weak linear correlation with the target; missing values in weight and market_index.
- Time: day-to-day noise; monthly medians more stable (Jan–Oct 2025).
- Validation: same feature schema; similar missing values; **city categories not seen in train** → encoding should handle unknowns if labels are used.
- Time: train-test ends in Oct, validation is Nov–Dec → a time-based split on train-test makes sense.
- December: fewer columns than train/validation (no lat/lon); the pipeline must run with those columns.